# 01 — Data Cleaning: Asthma Disease Dataset (Patient-Level)

**Input:** `data/raw/asthma_disease_data.csv` (2,392 patients, 29 columns)
**Goal:** validate, clean, and feature-engineer the patient-level dataset so it's ready for
EDA, Machine Learning, and the Excel/Power BI/Tableau dashboards.

> **Version note:** this notebook was rebuilt on a revised, more realistic version of the
> dataset. Unlike the original file (where almost no feature was statistically associated with
> the diagnosis), this version shows a near-balanced diagnosis rate (~49%) and strong, clinically
> plausible associations between diagnosis and smoking, allergy history, symptoms, and lung
> function — confirmed later in `03_EDA_Patient.ipynb`.


In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 50)

RAW_PATH = "../data/raw/asthma_disease_data.csv"
df = pd.read_csv(RAW_PATH)

print("Shape:", df.shape)
df.head()


Shape: (2392, 29)


,PatientID,Age,Gender,Ethnicity,EducationLevel,BMI,Smoking,PhysicalActivity,DietQuality,SleepQuality,PollutionExposure,PollenExposure,DustExposure,PetAllergy,FamilyHistoryAsthma,HistoryOfAllergies,Eczema,HayFever,GastroesophagealReflux,LungFunctionFEV1,LungFunctionFVC,Wheezing,ShortnessOfBreath,ChestTightness,Coughing,NighttimeSymptoms,ExerciseInduced,Diagnosis,DoctorInCharge
0,5034,11,1,0,3,24.086439,0,0.992883,3.033482,4.353554,4.569629,6.845372,6.061132,1,1,0,0,1,0,2.475998,3.299287,0,1,1,1,1,1,1,Dr_Confid
1,5035,63,0,0,3,27.729302,1,4.442662,6.894378,6.215557,4.454586,3.960348,7.245001,0,0,1,0,0,0,2.032021,2.616586,1,1,0,1,1,0,1,Dr_Confid
2,5036,54,0,1,1,21.865899,1,0.403755,3.587354,7.641393,6.273242,4.071867,7.175497,0,0,1,0,1,1,2.044598,2.899849,0,1,1,0,1,0,1,Dr_Confid
3,5037,38,1,3,2,23.816142,0,1.424408,3.658167,5.901292,6.586813,6.970949,1.799196,0,0,1,0,0,1,3.325531,3.695034,0,0,0,0,0,0,0,Dr_Confid
4,5038,37,1,0,1,24.399613,0,5.424333,7.108879,6.687806,6.291633,5.507114,6.746308,0,0,0,0,0,0,2.376773,2.948967,1,1,1,0,0,1,1,Dr_Confid


## 1. Initial inspection

In [2]:
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 2392 entries, 0 to 2391
Data columns (total 29 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   PatientID               2392 non-null   int64  
 1   Age                     2392 non-null   int64  
 2   Gender                  2392 non-null   int64  
 3   Ethnicity               2392 non-null   int64  
 4   EducationLevel          2392 non-null   int64  
 5   BMI                     2392 non-null   float64
 6   Smoking                 2392 non-null   int64  
 7   PhysicalActivity        2392 non-null   float64
 8   DietQuality             2392 non-null   float64
 9   SleepQuality            2392 non-null   float64
 10  PollutionExposure       2392 non-null   float64
 11  PollenExposure          2392 non-null   float64
 12  DustExposure            2392 non-null   float64
 13  PetAllergy              2392 non-null   int64  
 14  FamilyHistoryAsthma     2392 non-null   int64  
 15

In [3]:
df.describe(include='all').T


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
PatientID,2392.0,NaN,NaN,NaN,6229.5,690.655244,5034.0,5631.75,6229.5,6827.25,7425.0
Age,2392.0,NaN,NaN,NaN,42.536789,21.971397,5.0,23.0,42.0,62.0,80.0
Gender,2392.0,NaN,NaN,NaN,0.483278,0.499825,0.0,0.0,0.0,1.0,1.0
Ethnicity,2392.0,NaN,NaN,NaN,0.782609,1.023144,0.0,0.0,0.0,1.0,3.0
EducationLevel,2392.0,NaN,NaN,NaN,1.630017,0.891327,0.0,1.0,2.0,2.0,3.0
BMI,2392.0,NaN,NaN,NaN,26.132072,5.008833,15.0,22.686701,26.00358,29.545922,40.0
Smoking,2392.0,NaN,NaN,NaN,0.199833,0.399958,0.0,0.0,0.0,0.0,1.0
PhysicalActivity,2392.0,NaN,NaN,NaN,4.743821,2.184698,0.0,3.268477,4.702205,6.175809,10.0
DietQuality,2392.0,NaN,NaN,NaN,5.477532,1.99432,0.0,4.085725,5.436621,6.860425,10.0
SleepQuality,2392.0,NaN,NaN,NaN,6.989175,1.284466,4.0,6.117501,6.999031,7.870293,10.0


## 2. Data quality checks

### 2.1 Missing values

In [4]:
missing = df.isnull().sum()
missing = missing[missing > 0]
print("Columns with missing values:")
print(missing if len(missing) else "None — dataset has no missing values.")


Columns with missing values:
None — dataset has no missing values.


### 2.2 Duplicate rows / duplicate PatientIDs

In [5]:
print("Fully duplicated rows:", df.duplicated().sum())
print("Duplicate PatientIDs:", df['PatientID'].duplicated().sum())


Fully duplicated rows: 0
Duplicate PatientIDs: 0


### 2.3 Range / sanity checks

In [6]:
expected_ranges = {
    'Age': (5, 80),
    'BMI': (15, 40),
    'PhysicalActivity': (0, 10),
    'DietQuality': (0, 10),
    'SleepQuality': (4, 10),
    'PollutionExposure': (0, 10),
    'PollenExposure': (0, 10),
    'DustExposure': (0, 10),
    'LungFunctionFEV1': (1.0, 4.0),
    'LungFunctionFVC': (1.5, 6.0),
}

for col, (lo, hi) in expected_ranges.items():
    actual_lo, actual_hi = df[col].min(), df[col].max()
    status = "OK" if actual_lo >= lo and actual_hi <= hi else "CHECK"
    print(f"{col:22s} expected [{lo}, {hi}]  actual [{actual_lo:.2f}, {actual_hi:.2f}]  -> {status}")


Age                    expected [5, 80]  actual [5.00, 80.00]  -> OK
BMI                    expected [15, 40]  actual [15.00, 40.00]  -> OK
PhysicalActivity       expected [0, 10]  actual [0.00, 10.00]  -> OK
DietQuality            expected [0, 10]  actual [0.00, 10.00]  -> OK
SleepQuality           expected [4, 10]  actual [4.00, 10.00]  -> OK
PollutionExposure      expected [0, 10]  actual [0.00, 10.00]  -> OK
PollenExposure         expected [0, 10]  actual [0.00, 10.00]  -> OK
DustExposure           expected [0, 10]  actual [0.00, 10.00]  -> OK
LungFunctionFEV1       expected [1.0, 4.0]  actual [1.07, 4.00]  -> OK
LungFunctionFVC        expected [1.5, 6.0]  actual [1.50, 5.90]  -> OK


### 2.4 Binary / categorical columns — check for unexpected codes

In [7]:
binary_cols = ['Gender','Smoking','PetAllergy','FamilyHistoryAsthma','HistoryOfAllergies',
               'Eczema','HayFever','GastroesophagealReflux','Wheezing','ShortnessOfBreath',
               'ChestTightness','Coughing','NighttimeSymptoms','ExerciseInduced','Diagnosis']

for col in binary_cols:
    vals = sorted(df[col].unique())
    flag = "OK" if set(vals) <= {0, 1} else "CHECK"
    print(f"{col:25s} unique values: {vals}  -> {flag}")

print()
print("Ethnicity unique values:", sorted(df['Ethnicity'].unique()))
print("EducationLevel unique values:", sorted(df['EducationLevel'].unique()))


Gender                    unique values: [np.int64(0), np.int64(1)]  -> OK
Smoking                   unique values: [np.int64(0), np.int64(1)]  -> OK
PetAllergy                unique values: [np.int64(0), np.int64(1)]  -> OK
FamilyHistoryAsthma       unique values: [np.int64(0), np.int64(1)]  -> OK
HistoryOfAllergies        unique values: [np.int64(0), np.int64(1)]  -> OK
Eczema                    unique values: [np.int64(0), np.int64(1)]  -> OK
HayFever                  unique values: [np.int64(0), np.int64(1)]  -> OK
GastroesophagealReflux    unique values: [np.int64(0), np.int64(1)]  -> OK
Wheezing                  unique values: [np.int64(0), np.int64(1)]  -> OK
ShortnessOfBreath         unique values: [np.int64(0), np.int64(1)]  -> OK
ChestTightness            unique values: [np.int64(0), np.int64(1)]  -> OK
Coughing                  unique values: [np.int64(0), np.int64(1)]  -> OK
NighttimeSymptoms         unique values: [np.int64(0), np.int64(1)]  -> OK
ExerciseInduced          

### 2.5 Target balance — a first look

In [8]:
print(df['Diagnosis'].value_counts())
print((df['Diagnosis'].value_counts(normalize=True) * 100).round(2))


Diagnosis
0    1221
1    1171
Name: count, dtype: int64
Diagnosis
0    51.05
1    48.95
Name: proportion, dtype: float64


## 3. Drop irrelevant columns

`DoctorInCharge` is constant (`"Dr_Confid"`) for every patient — no information, confidential,
dropped.

In [9]:
print("Unique DoctorInCharge values:", df['DoctorInCharge'].unique())
df = df.drop(columns=['DoctorInCharge'])
df.shape


Unique DoctorInCharge values: <StringArray>
['Dr_Confid']
Length: 1, dtype: str


(2392, 28)

## 4. Add human-readable label columns

Kept the original numeric codes (for modeling) and added matching `*_Label` columns (for
dashboards, slicers, and readable chart legends).

In [10]:
df['Gender_Label'] = df['Gender'].map({0: 'Male', 1: 'Female'})

df['Ethnicity_Label'] = df['Ethnicity'].map({
    0: 'Caucasian', 1: 'African American', 2: 'Asian', 3: 'Other'
})

df['EducationLevel_Label'] = df['EducationLevel'].map({
    0: 'None', 1: 'High School', 2: "Bachelor's", 3: 'Higher'
})

yes_no_cols = ['Smoking','PetAllergy','FamilyHistoryAsthma','HistoryOfAllergies','Eczema',
               'HayFever','GastroesophagealReflux','Wheezing','ShortnessOfBreath',
               'ChestTightness','Coughing','NighttimeSymptoms','ExerciseInduced']

for col in yes_no_cols:
    df[col + '_Label'] = df[col].map({0: 'No', 1: 'Yes'})

df['Diagnosis_Label'] = df['Diagnosis'].map({0: 'Negative', 1: 'Positive'})

df.filter(like='_Label').head()


,Gender_Label,Ethnicity_Label,EducationLevel_Label,Smoking_Label,PetAllergy_Label,FamilyHistoryAsthma_Label,HistoryOfAllergies_Label,Eczema_Label,HayFever_Label,GastroesophagealReflux_Label,Wheezing_Label,ShortnessOfBreath_Label,ChestTightness_Label,Coughing_Label,NighttimeSymptoms_Label,ExerciseInduced_Label,Diagnosis_Label
0,Female,Caucasian,Higher,No,Yes,Yes,No,No,Yes,No,No,Yes,Yes,Yes,Yes,Yes,Positive
1,Male,Caucasian,Higher,Yes,No,No,Yes,No,No,No,Yes,Yes,No,Yes,Yes,No,Positive
2,Male,African American,High School,Yes,No,No,Yes,No,Yes,Yes,No,Yes,Yes,No,Yes,No,Positive
3,Female,Other,Bachelor's,No,No,No,Yes,No,No,Yes,No,No,No,No,No,No,Negative
4,Female,Caucasian,High School,No,No,No,No,No,No,No,Yes,Yes,Yes,No,No,Yes,Positive


## 5. Feature engineering

In [11]:
def age_group(age):
    if age < 18: return '0-17 (Child)'
    if age < 30: return '18-29'
    if age < 45: return '30-44'
    if age < 60: return '45-59'
    return '60+'

df['AgeGroup'] = df['Age'].apply(age_group)

def bmi_category(bmi):
    if bmi < 18.5: return 'Underweight'
    if bmi < 25: return 'Normal'
    if bmi < 30: return 'Overweight'
    return 'Obese'

df['BMICategory'] = df['BMI'].apply(bmi_category)

df['FEV1_FVC_Ratio'] = (df['LungFunctionFEV1'] / df['LungFunctionFVC']).round(3)

symptom_cols = ['Wheezing','ShortnessOfBreath','ChestTightness','Coughing',
                'NighttimeSymptoms','ExerciseInduced']
df['SymptomCount'] = df[symptom_cols].sum(axis=1)

risk_cols = ['FamilyHistoryAsthma','HistoryOfAllergies','Eczema','HayFever',
             'GastroesophagealReflux']
df['RiskFactorCount'] = df[risk_cols].sum(axis=1)

df[['Age','AgeGroup','BMI','BMICategory','FEV1_FVC_Ratio','SymptomCount','RiskFactorCount']].head()


,Age,AgeGroup,BMI,BMICategory,FEV1_FVC_Ratio,SymptomCount,RiskFactorCount
0,11,0-17 (Child),24.086439,Normal,0.750,5,2
1,63,60+,27.729302,Overweight,0.777,4,1
2,54,45-59,21.865899,Normal,0.705,3,3
3,38,30-44,23.816142,Normal,0.900,0,2
4,37,30-44,24.399613,Normal,0.806,4,0


## 6. Tidy up numeric precision

In [12]:
float_cols = df.select_dtypes(include='float').columns
df[float_cols] = df[float_cols].round(2)
df.head()


,PatientID,Age,Gender,Ethnicity,EducationLevel,BMI,Smoking,PhysicalActivity,DietQuality,SleepQuality,PollutionExposure,PollenExposure,DustExposure,PetAllergy,FamilyHistoryAsthma,HistoryOfAllergies,Eczema,HayFever,GastroesophagealReflux,LungFunctionFEV1,LungFunctionFVC,Wheezing,ShortnessOfBreath,ChestTightness,Coughing,NighttimeSymptoms,ExerciseInduced,Diagnosis,Gender_Label,Ethnicity_Label,EducationLevel_Label,Smoking_Label,PetAllergy_Label,FamilyHistoryAsthma_Label,HistoryOfAllergies_Label,Eczema_Label,HayFever_Label,GastroesophagealReflux_Label,Wheezing_Label,ShortnessOfBreath_Label,ChestTightness_Label,Coughing_Label,NighttimeSymptoms_Label,ExerciseInduced_Label,Diagnosis_Label,AgeGroup,BMICategory,FEV1_FVC_Ratio,SymptomCount,RiskFactorCount
0,5034,11,1,0,3,24.09,0,0.99,3.03,4.35,4.57,6.85,6.06,1,1,0,0,1,0,2.48,3.30,0,1,1,1,1,1,1,Female,Caucasian,Higher,No,Yes,Yes,No,No,Yes,No,No,Yes,Yes,Yes,Yes,Yes,Positive,0-17 (Child),Normal,0.75,5,2
1,5035,63,0,0,3,27.73,1,4.44,6.89,6.22,4.45,3.96,7.25,0,0,1,0,0,0,2.03,2.62,1,1,0,1,1,0,1,Male,Caucasian,Higher,Yes,No,No,Yes,No,No,No,Yes,Yes,No,Yes,Yes,No,Positive,60+,Overweight,0.78,4,1
2,5036,54,0,1,1,21.87,1,0.40,3.59,7.64,6.27,4.07,7.18,0,0,1,0,1,1,2.04,2.90,0,1,1,0,1,0,1,Male,African American,High School,Yes,No,No,Yes,No,Yes,Yes,No,Yes,Yes,No,Yes,No,Positive,45-59,Normal,0.70,3,3
3,5037,38,1,3,2,23.82,0,1.42,3.66,5.90,6.59,6.97,1.80,0,0,1,0,0,1,3.33,3.70,0,0,0,0,0,0,0,Female,Other,Bachelor's,No,No,No,Yes,No,No,Yes,No,No,No,No,No,No,Negative,30-44,Normal,0.90,0,2
4,5038,37,1,0,1,24.40,0,5.42,7.11,6.69,6.29,5.51,6.75,0,0,0,0,0,0,2.38,2.95,1,1,1,0,0,1,1,Female,Caucasian,High School,No,No,No,No,No,No,No,Yes,Yes,Yes,No,No,Yes,Positive,30-44,Normal,0.81,4,0


## 7. Final checks before export

In [13]:
print("Final shape:", df.shape)
print("Any nulls left?", df.isnull().sum().sum())
print()
print(df['Diagnosis_Label'].value_counts())
print()
print(df.columns.tolist())


Final shape: (2392, 50)
Any nulls left? 0

Diagnosis_Label
Negative    1221
Positive    1171
Name: count, dtype: int64

['PatientID', 'Age', 'Gender', 'Ethnicity', 'EducationLevel', 'BMI', 'Smoking', 'PhysicalActivity', 'DietQuality', 'SleepQuality', 'PollutionExposure', 'PollenExposure', 'DustExposure', 'PetAllergy', 'FamilyHistoryAsthma', 'HistoryOfAllergies', 'Eczema', 'HayFever', 'GastroesophagealReflux', 'LungFunctionFEV1', 'LungFunctionFVC', 'Wheezing', 'ShortnessOfBreath', 'ChestTightness', 'Coughing', 'NighttimeSymptoms', 'ExerciseInduced', 'Diagnosis', 'Gender_Label', 'Ethnicity_Label', 'EducationLevel_Label', 'Smoking_Label', 'PetAllergy_Label', 'FamilyHistoryAsthma_Label', 'HistoryOfAllergies_Label', 'Eczema_Label', 'HayFever_Label', 'GastroesophagealReflux_Label', 'Wheezing_Label', 'ShortnessOfBreath_Label', 'ChestTightness_Label', 'Coughing_Label', 'NighttimeSymptoms_Label', 'ExerciseInduced_Label', 'Diagnosis_Label', 'AgeGroup', 'BMICategory', 'FEV1_FVC_Ratio', 'SymptomCo

## 8. Export cleaned data

In [14]:
df.to_csv('../data/processed/asthma_cleaned.csv', index=False)
df.to_excel('../data/processed/asthma_cleaned.xlsx', index=False, sheet_name='CleanData')

print("Saved asthma_cleaned.csv and asthma_cleaned.xlsx")


Saved asthma_cleaned.csv and asthma_cleaned.xlsx
